# Downstream Analysis

This tutorial builds on the quickstarts ({doc}`single <zero_shot_quickstart>` / {doc}`multi-sample <zero_shot_quickstart_multisample>`) and shows the analyses TERRA embeddings unlock: **gene-level embeddings**, **spatial gene-pair scoring**, **EMD-based spatial structure**, **subsetting**, and **in-silico perturbation**.

:::{note}
**TERRA requires an NVIDIA GPU** for the embedding step. This page ships with its outputs cleared — run it on a GPU machine to reproduce the figures.
:::

## 1. Setup, data, and embeddings

We reload the demo dataset and run the pipeline once, this time also saving the tokenized dataset to disk — the downstream functions operate on that tokenized `dataset`.

In [ ]:
import logging

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import torch
from datasets import load_from_disk

from terra import (download_pretrained, embed_dataset, get_average_gene_embed,
                   get_emd_distance, get_gene_embed, get_spatial_score,
                   harmonize_tokenize_embed_pipeline, perturb_dataset)
from terra.utils.evaluation import get_top_gene_pairs, get_top_gene_score

logging.basicConfig(level="INFO")
sc.settings.set_figure_params(dpi=80, frameon=False)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# Download the "Xenium Prime FFPE Human Skin" output bundle from 10x Genomics
# (https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-skin), unzip it,
# and point `xenium_dir` at the unzipped folder.
xenium_dir = "xenium_prime_human_skin"  # <-- edit to your unzipped bundle

# Cell-by-gene counts. The Xenium matrix also contains control probes / blank
# codewords; keep only real gene-expression features.
adata = sc.read_10x_h5(f"{xenium_dir}/cell_feature_matrix.h5")
adata.var_names_make_unique()
adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()

# Attach single-cell spatial coordinates from the Xenium cell metadata.
cells = pd.read_parquet(f"{xenium_dir}/cells.parquet").set_index("cell_id")
adata.obs = adata.obs.join(cells)
adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()

# TERRA needs a per-cell id and a sample/batch column.
adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["sample"] = "skin"

# Crop a contiguous tissue window so spatial neighborhoods stay intact while
# keeping the tutorial fast. Widen `window` (microns) for more cells.
xy = adata.obsm["spatial"]
x0, y0 = xy.min(0)
window = 750
keep = (xy[:, 0] <= x0 + window) & (xy[:, 1] <= y0 + window)
adata = adata[keep].copy()
print(adata)

In [ ]:
model_dir = download_pretrained("Lotfollahi-lab/TERRA-96M")

adata = harmonize_tokenize_embed_pipeline(
    adata=adata,
    sample_key=None,
    batch_key="sample",
    model_folder_path=model_dir,
    cache_directory_path="./terra_cache",
    save_dataset_path="./terra_cache/skin.dataset",
)

# The tokenized dataset that the downstream functions consume.
dataset = load_from_disk("./terra_cache/skin.dataset")

# Ensembl IDs of the panel genes (added to adata.var by harmonization).
gene_ids = list(adata.var["ensembl_id"])

## 2. Average gene embeddings

`get_average_gene_embed` returns, for each requested gene, its mean cell- and neighborhood-level embedding across the dataset. Embedding the genes themselves shows which ones group together.

In [ ]:
avg = get_average_gene_embed(
    dataset=dataset,
    model_folder_path=model_dir,
    cell_gene_ensembl_id=gene_ids,
    neighborhood_gene_ensembl_id=gene_ids,
)

gene_emb = np.asarray(avg["cell_gene_emb_average_per_data"])
ok = ~np.isnan(gene_emb).any(axis=1)
genes = sc.AnnData(gene_emb[ok])
genes.obs_names = np.asarray(gene_ids, dtype=str)[ok]

sc.pp.neighbors(genes, use_rep="X", n_neighbors=15)
sc.tl.umap(genes)
sc.tl.leiden(genes, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
sc.pl.umap(genes, color="leiden", title="Gene embedding UMAP")

## 3. Cell-level gene embeddings

`get_gene_embed` returns a per-cell embedding for specific genes. Replace the example Ensembl IDs with genes of interest from your panel.

:::{note}
The released models use Ensembl release 110 — look gene symbols up against that release for correct IDs.
:::

In [ ]:
genes_of_interest = gene_ids[:2]  # <-- replace with your genes of interest

gene_embed = get_gene_embed(
    dataset=dataset,
    model_folder_path=model_dir,
    cell_gene_ensembl_id=genes_of_interest,
    neighborhood_gene_ensembl_id=genes_of_interest,
)
for key, values in gene_embed.items():
    adata.obsm[key] = values
print(list(gene_embed))

## 4. Spatial gene-pair score

`get_spatial_score` compares each gene's cell vs. neighborhood representation; the ratio highlights genes whose signal is spatially structured. `get_top_gene_score` / `get_top_gene_pairs` rank the strongest genes and gene pairs.

In [ ]:
score = get_spatial_score(
    dataset=dataset,
    model_folder_path=model_dir,
    cell_gene_ensembl_id=gene_ids,
    neighborhood_gene_ensembl_id=gene_ids,
)
gene_pair_score = score["cos_sim_neighborhood"] / score["cos_sim_cell"]

df_scores = get_top_gene_score(
    gene_pair_score,
    cell_gene_ensembl_id=gene_ids,
    gene_df=adata.var,
    gene_counts=score["cell_count_neighborhood"],
    min_count=20,
)
df_scores.sort_values("gene_score", ascending=False).head(20)

In [ ]:
df_pairs = get_top_gene_pairs(
    torch.from_numpy(gene_pair_score),
    count_cell=torch.from_numpy(score["cell_count_cell"]),
    count_neb=torch.from_numpy(score["cell_count_neighborhood"]),
    cos_sim_cell=torch.from_numpy(score["cos_sim_cell"]),
    cos_sim_neb=torch.from_numpy(score["cos_sim_neighborhood"]),
    gene_df=adata.var,
    cell_gene_ids=gene_ids,
    neighborhood_gene_ids=gene_ids,
    min_count=20,
    k=100,
)
df_pairs.head(20)

## 5. EMD-based spatial structure

`get_emd_distance` scores how spatially structured each cell's neighborhood is; plotting it in space highlights organized regions.

In [ ]:
adata.obs["emd_dist"] = get_emd_distance(
    dataset=dataset,
    model_folder_path=model_dir,
    cell_gene_ensembl_id=gene_ids,
    neighborhood_gene_ensembl_id=gene_ids,
)
sq.pl.spatial_scatter(adata, color="emd_dist", shape=None, size=6)

## 6. Analyze a subset of cells

You can restrict the tokenized dataset to a set of cell IDs (e.g. a niche or cell type from a quickstart) and repeat any analysis on it.

In [ ]:
subset_ids = set(map(str, adata.obs["cell_id"][:1000]))
subset = dataset.filter(lambda ex: str(ex["cell_id"]) in subset_ids, num_proc=1)
print(subset)

## 7. In-silico perturbation

`perturb_dataset` edits gene tokens (knockout or fold-change) for chosen cells; re-embedding the perturbed dataset shows the effect in latent space.

In [ ]:
example_cell = str(adata.obs["cell_id"].iloc[0])
perturb_df = pd.DataFrame({
    "perturbed_cell_id": [example_cell, example_cell],
    "perturbed_ensembl_id": ["all", gene_ids[0]],
    "perturbation_target": ["cell", "cell"],
    "perturbation_type": ["knockout", "foldchange"],
    "foldchange": [np.nan, 0.5],
})

perturbed = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    model_folder_path=model_dir,
)
perturbed_emb = embed_dataset(dataset=perturbed, model_folder_path=model_dir)
adata.obsm["perturbed_cell_emb"] = perturbed_emb["cell_emb"]
adata.obsm["perturbed_neighborhood_emb"] = perturbed_emb["neighborhood_emb"]

## Next steps

See the {doc}`../api` for the full reference of every function used here, and the {doc}`../user_guide` for the concepts behind the embeddings.